# Liquidity Management Calibrations

References for this notebook can be found [here](../../docs/notebooks_notes/liquidity_references.md)
1. **Liquidity profile** — days-to-liquidate, bucket distribution, redemption stress, investor concentration, liquidity-adjusted VaR
2. **LMT trigger analysis** — 12-month gate / swing / suspension simulation per AIFMD II Art. 16a

In [ ]:
from src.data.database   import get_engine
from src.risk.risk_utils import redemption_stress

ENGINE  = get_engine()

In [ ]:
# in teis repo asll computation refer to the same date
from src.config import VALUATION_DATE, LIQUIDITY_BUCKET_ORDER
VALUATION_DATE

### 1. Liquidity Profile

Days-to-liquidate and ESMA bucket distribution across all six funds. Positions loaded from the database at the standard valuation date.


In [ ]:
# Liquidity profile for the selected fund
from src.data.database import query_positions
from src.data.reference_data import get_fund_name
from src.risk.risk_utils import compute_liquidity_profile
from src.ui.liquidity_calibration_display import plot_liquidity_profile_chart

SELECTED_FUND = 'UCITS_Balanced'

# Query and enrich positions
from src.data.enrichment import get_risk_ready_df
risk_df = get_risk_ready_df(ENGINE, SELECTED_FUND, VALUATION_DATE)
nav = risk_df['market_value_eur'].sum()

# Compute liquidity profile using the canonical function
liq = compute_liquidity_profile(risk_df, nav, pct_adv=0.25)
risk_df_liq = liq['risk_df_liq']
bucket_full = liq['bucket_full']

# Plot by fund
funds_data = {get_fund_name(SELECTED_FUND): risk_df_liq}
plot_liquidity_profile_chart(funds_data, VALUATION_DATE, LIQUIDITY_BUCKET_ORDER, export_id="01")


In [ ]:
# Load investor registers and calibration inputs
from src.data.reference_data import (
    load_investor_and_calibration_data,
)
from src.computation.liquidity_calibration import (
    compute_redemption_scenarios,
)

# Load calibration data for UCITS_Balanced
data = load_investor_and_calibration_data(SELECTED_FUND)
investor_inputs = data['investor_inputs']
calibration_inputs = data['calibration_inputs']
calibration_config = data['calibration_config']

# Compute redemption scenarios
scenarios_result = compute_redemption_scenarios(investor_inputs, calibration_config)

print(f"✓ Loaded investor registers and computed redemption scenarios for {SELECTED_FUND}")


#### 1.2 Redemption Stress — Single-Period Snapshot

Point-in-time gate analysis: can each open-ended fund meet a 25% redemption within a 5-day notice period? 

In [ ]:
# Redemption stress testing
from src.ui.liquidity_calibration_display import display_redemption_stress_table
from src.data.reference_data import get_stress_testing_params, get_fund_name
from src.risk.risk_utils import redemption_stress, compute_liquidity_profile

# Get stress testing params from calibration
params = get_stress_testing_params(SELECTED_FUND, calibration_inputs, redemption_scenario="Large")

# Query positions for selected fund
pos = query_positions(ENGINE, SELECTED_FUND, VALUATION_DATE)
nav = pos['market_value_eur'].sum()

# Compute liquidity profile and stress scenario
liq = compute_liquidity_profile(pos, nav, pct_adv=params['pct_adv'])
rs = redemption_stress(liq['risk_df_liq'], nav, redemption_pct=params['redemption_pct'], notice_days=params['notice_days'])

fund_name = get_fund_name(SELECTED_FUND)
stress_results = {
    fund_name: {
        'nav': nav,
        'redemption_amount_eur': rs['redemption_amount_eur'],
        'liquid_assets_eur': rs['liquid_assets_eur'],
        'coverage_ratio': rs['coverage_ratio'],
        'can_meet_redemption': rs['can_meet_redemption'],
    }
}

display_redemption_stress_table(stress_results, VALUATION_DATE, export_id="02")


### 2. Investor Base Model – Redemption Schedule

The redemption schedule is derived from a fund-level **investor register** and calibration inputs. Each investor type has an AUM share and separate redemption assumptions for normal and stressed conditions.

| Notation | Meaning |
|---|---|
| $w_i$ | AUM share of investor type $i$ ($\sum_i w_i = 1$) |
| $\mu_i^{\text{norm}}$ | Monthly redemption rate under normal conditions |
| $\mu_i^{\text{stress}}$ | Monthly redemption rate under stress conditions |
| $\kappa$ | Concentration parameter controlling dispersion around the normal rate |

#### Normal months – beta-distributed draws

For months not designated as stress months, each investor type draws from a beta distribution centred on its normal redemption rate. The beta distribution is used because:

- It keeps redemption rates strictly bounded between 0 and 1
- The concentration parameter $\kappa$ controls dispersion symmetrically around the mean
- Higher $\kappa$ produces more concentrated (stable) monthly draws
- Lower $\kappa$ produces more dispersed (volatile) draws

The beta parameters are derived from the normal rate and concentration:

$$\alpha_i = \mu_i^{\text{norm}} \cdot \kappa$$

$$\beta_i = (1 - \mu_i^{\text{norm}}) \cdot \kappa$$

With this parametrisation:

$$X_{i,t} \sim \text{Beta}(\alpha_i, \beta_i), \quad \mathbb{E}[X_{i,t}] = \mu_i^{\text{norm}}$$

The aggregate monthly redemption rate is the AUM-weighted sum across investor types:

$$r_t = \sum_{i=1}^{N} w_i \cdot X_{i,t}$$

#### Stress months – deterministic override

For designated stress months $t \in \mathcal{S}$, the model uses the stressed redemption rate directly:

$$r_t = \sum_{i=1}^{N} w_i \cdot \mu_i^{\text{stress}}$$

This represents a deterministic stress scenario rather than a random draw. The stress months and stressed rates are fund-level calibration inputs.

#### Reference rates

The normal and stressed reference rates are the weighted averages:

$$r^{\text{norm,wt}} = \sum_i w_i \cdot \mu_i^{\text{norm}}$$

$$r^{\text{stress,wt}} = \sum_i w_i \cdot \mu_i^{\text{stress}}$$

These provide a benchmark for the simulated redemption path.

### 3. Liquidity Management Tools (LMT)

This section models fund-level liquidity management tools as described in fund risk policy and applied within a 12-month stochastic redemption simulation.


#### Three LMTs modelled

| Tool | Type | Trigger condition | Action |
|---|---|---|---|
| Redemption gate | Quantitative | Liquid NAV falls below gate threshold | Cap redemptions paid; defer excess |
| Swing pricing | Anti-dilution | Aggregate redemption rate exceeds threshold | Apply cost adjustment to exiting investors |
| Suspension | Quantitative | Consecutive gate months or backlog breach | Suspend redemptions temporarily |

#### Gate trigger

A **redemption gate** is activated when the fund's liquid NAV falls below a fund-policy gate threshold (as a % of total NAV).

When a gate is active:
- Redemption requests up to the liquid buffer are met in full
- Requests exceeding the liquid buffer are **deferred** (carried forward to the next month)
- The deferred amount accumulates as a **backlog**

#### Swing pricing

**Swing pricing** applies an anti-dilution cost to redeeming investors when aggregate redemptions exceed a threshold. This protects remaining shareholders from bearing the transaction costs of redemptions.

Implementation:
- When swing is triggered, the redemption price is adjusted downward by a swing levy (e.g., 0.5% of NAV)
- This cost adjustment is borne by redeeming investors, not by the fund balance


#### Suspension

A **suspension** triggers when:
- A gate has been active for a specified number of consecutive months (e.g., 3 months), or
- The deferred backlog exceeds a threshold relative to liquid NAV (e.g., 25%)

When suspended, all new redemption requests are deferred until the trigger condition clears.

#### 3.1 UCITS Balanced

**Portfolio**: multi-asset (equities, IG/HY bonds, listed alternatives). Liquid sleeve estimated at **85% of NAV** — the remaining 15% is allocated to less-liquid instruments (HY bonds and listed alternatives with lower daily turnover).

**Parameters**:

| Parameter | Value | Rationale |
|---|---|---|
| Gate threshold $G$ | 10% of NAV | Standard UCITS gate; contractual fund document limit |
| Swing threshold $S$ | 5% of NAV | Activates when net flows are large enough to impose material transaction costs |
| Swing factor $\delta$ | 50 bps | Market impact estimate for the liquid sleeve |
| Contagion multiplier $\lambda$ | 1.3 | Mixed retail/institutional base; slower reaction than pure HF |
| Suspension conditions | $N = 3$ consecutive gate months; $\theta_B = 25\%$ | Conservative — UCITS retail investor protections favour later, deliberate board decision |

**Stress schedule**: two consecutive spike months (months 3–4 at 10–14% of NAV) followed by sustained moderate outflow. Total 12-month gross request ≈ 61% of initial NAV.

In [ ]:
# UCITS Balanced — LMT Trigger Analysis

from src.ui.liquidity_calibration_display import (
    display_redemption_scenarios,
    display_lmt_summary,
    plot_lmt_analysis,
)
from src.data.reference_data import get_lmt_parameters, build_lmt_parameters
from src.risk.risk_utils import lmt_trigger_analysis

# Build LMT parameters
lmt_params = build_lmt_parameters(SELECTED_FUND, calibration_inputs, calibration_config)
lmt_config = get_lmt_parameters(SELECTED_FUND, calibration_inputs)

# Query positions and NAV
pos = query_positions(ENGINE, SELECTED_FUND, VALUATION_DATE)
nav = pos['market_value_eur'].sum()

# Run LMT trigger analysis
df_analysis = lmt_trigger_analysis(
    nav=nav,
    liquid_pct=lmt_config['liquid_pct'],
    gate_threshold=lmt_config['gate_threshold'],
    swing_threshold=lmt_config['swing_threshold'],
    redemption_schedule=lmt_params['schedule'],
    consecutive_gate_for_suspension=lmt_config['consec_gate'],
    backlog_pct_for_suspension=lmt_config['backlog_pct'],
    contagion_multiplier=lmt_config['contagion'],
)

# Display results
scenarios_data = {
    'redemption_scenarios': scenarios_result['redemption_scenarios'],
    'largest_investor_name': scenarios_result['largest_investor_name'],
    'largest_investor_pct': scenarios_result['largest_investor_pct'],
}

display_redemption_scenarios(scenarios_data, SELECTED_FUND, VALUATION_DATE)
display_lmt_summary(df_analysis, SELECTED_FUND, VALUATION_DATE)
plot_lmt_analysis(df_analysis, SELECTED_FUND, VALUATION_DATE)


In [ ]:
# Suggested liquidity monitoring block — UCITS
from src.pipeline.liquidity_policy import build_fund_liquidity_policy, export_liquidity_policy
from src.ui.liquidity_calibration_display import display_suggested_policy_block
from src.data.reference_data import get_fund_name

# Query NAV for the selected fund
pos = query_positions(ENGINE, SELECTED_FUND, VALUATION_DATE)
nav = pos['market_value_eur'].sum()

# Build policy block
policy = build_fund_liquidity_policy(
    SELECTED_FUND,
    calibration_inputs,
    scenarios_result,
    nav,
)

# Display and optionally export
display_suggested_policy_block(policy, get_fund_name(SELECTED_FUND))

# Uncomment to export to file:
# export_path = export_liquidity_policy(SELECTED_FUND, policy)
# print(f"✓ Policy exported to {export_path}")
